# tk-assistant · QLoRA-обучение на T4 (Этап 3)

Требования к запуску:
- датасет в репо (`data/processed/train.jsonl`, `val.jsonl` — этап 1 завершён);
- Colab runtime → **T4 GPU**.

План экспериментов (каждый — отдельная запись в W&B и чекпоинт):
1. epochs=1, lr=2e-4 (базовый, `configs/train_qlora.yaml`)
2. epochs=2, lr=2e-4
3. epochs=3, lr=2e-4
4. лучший по val → сравнить lr 1e-4 vs 2e-4

После каждого прогона: просмотреть 10 сэмплов (скрипт печатает в конце) и записать в W&B заметку о качестве.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%capture
!pip install unsloth wandb rouge-score

## 1. Код проекта и данные

In [ ]:
%%capture
!git clone https://github.com/markkentos/tk-assistant.git
%cd /content/tk-assistant

import json, pathlib
for name in ("train", "val"):
    path = pathlib.Path(f"data/processed/{name}.jsonl")
    n = sum(1 for _ in path.open(encoding="utf-8"))
    print(f"{name}: {n} пар")
    assert n > 0, f"{path} пуст — этап 1 не завершён"

## 2. Ключи (Colab Secrets или ввод вручную)

В Colab: иконка ключа слева → Secrets → добавь `WANDB_API_KEY` и `HF_TOKEN`.

## 2. Ключи (необязательно)

W&B заблокирован — трекинг идёт в **TensorBoard** (локально, без ключей).
`HF_TOKEN` нужен только для пуша адаптера на HF Hub: Colab → Secrets → `HF_TOKEN`.

## 2. Ключи (необязательно)

W&B заблокирован — трекинг идёт в **TensorBoard** (локально, без ключей).
`HF_TOKEN` нужен только для пуша адаптера на HF Hub: Colab → Secrets → `HF_TOKEN`.

In [ ]:
import os

def get_secret(name: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

os.environ["HF_TOKEN"] = get_secret("HF_TOKEN")
print("HF_TOKEN:", "задан" if os.environ["HF_TOKEN"] else "не задан (ок, push работать не будет)")

## 3. Обучение

Конфиг — `configs/train_qlora.yaml`. Для эксперимента №2: скопируй конфиг, поменяй `epochs: 2` и `output.adapter_dir`, запусти эту ячейку с новым конфигом.

In [ ]:
!python src/train_qlora.py --config configs/train_qlora.yaml

## 3.1 Графики обучения (TensorBoard)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/adapter_e1_lr2e-4/checkpoints

## 4. Сохранить адаптер

Вариант А: скачать zip на диск. Вариант Б: залить на HF Hub (`configs/train_qlora.yaml` → `output.push_to_hub: true`, `hf_repo: <username>/qwen2.5-7b-tk-qlora-e1`).

In [ ]:
!zip -r adapter_e1_lr2e-4.zip outputs/adapter_e1_lr2e-4 -x "*/checkpoints/*"
from google.colab import files
files.download("adapter_e1_lr2e-4.zip")

## 5. Ответы tuned-модели на бенчмарке (той же сессией, пока GPU живой)

In [ ]:
!python src/generate_answers.py --mode tuned --adapter outputs/adapter_e1_lr2e-4 --input eval/benchmark.jsonl --output eval/runs/answers_tuned.jsonl
files.download("eval/runs/answers_tuned.jsonl")

## 3.1 Графики обучения (TensorBoard)

## 5. Ответы tuned-модели на бенчмарке (той же сессией, пока GPU живой)

In [ ]:
!python src/generate_answers.py --mode tuned --adapter outputs/adapter_e1_lr2e-4 --input eval/benchmark.jsonl --output eval/runs/answers_tuned.jsonl
files.download("eval/runs/answers_tuned.jsonl")